Load environment variables using the dotenv extension, which allows the notebook to access configuration settings from .env files.

In [1]:
%load_ext dotenv
%dotenv 

# What are we doing?

## Objectives 


* Build a data pipeline that downloads price data from the internet, stores it locally, transforms it into return data, and stores the feature set.
    - Getting the data.
    - Schemas and index in dask.

* Explore the parquet format.
    - Reading and writing parquet files.
    - Read datasets that are stored in distributed files.
    - Discuss dask vs pandas as a small example of big vs small data.
    
* Discuss the use of environment variables for settings.
* Discuss how to use Jupyter notebooks and source code concurrently. 
* Logging and using a standard logger.

## About the Data

+ We will download the prices for a list of stocks.
+ The source is Yahoo Finance and we will use the API provided by the library yfinance.


## Medallion Architecture

+ The architecture that we are thinking about is called Medallion by [DataBricks](https://www.databricks.com/glossary/medallion-architecture). It is an ELT type of thinking, although our data is well-structured.

![Medallion Architecture (DataBicks)](./images/02_medallion_architecture.png)

+ In our case, we would like to optimize the number of times that we download data from the internet. 
+ Ultimately, we will build a pipeline manager class that will help us control the process of obtaining and transforming our data.

![](./images/02_target_pipeline_manager.png)

# Download Data

Download the [Stock Market Dataset from Kaggle](https://www.kaggle.com/datasets/jacksoncrow/stock-market-dataset). Note that you may be required to register for a free account.

Extract all files into the directory: `./05_src/data/prices_csv/`

Your folder structure should include the following paths:

+ `05_src/data/prices_csv/etfs`
+ `05_src/data/prices_csv/stocks`


In [2]:
import pandas as pd
import os
import sys
from glob import glob

sys.path.append(os.getenv('SRC_DIR'))

from utils.logger import get_logger
_logs = get_logger(__name__)

A few things to notice in the code chunk above:

+ Libraries are ordered from high-level to low-level libraries from the package manager (pip in this case, but could be conda, poetry, etc.)
+ The command `sys.path.append("../05_src/)` will add the `../05_src/` directory to the path in the Notebook's kernel. This way, we can use our modules as part of the notebook.
+ Local modules are imported at the end. 
+ The function `get_logger()` is called with `__name__` as recommended by the documentation.

Now, to load the historical price data for stocks and ETFs, we could use:

In [3]:
import random
# Load stock price CSV files from a specified directory
stock_files = glob(os.path.join(os.getenv('SRC_DIR'), "data/prices_csv/stocks/*.csv"))

# Select 60 random stock CSVs under a configured source directory
random.seed(42)
stock_files = random.sample(stock_files, 60)

# logs each read, adds metadata columns (source file and ticker), 
# converts the date column, and merges them into a single DataFrame for further analysis.
dt_list = []
for s_file in stock_files:
    _logs.info(f"Reading file: {s_file}")                       # Log the file being read
    dt = pd.read_csv(s_file).assign(                            # Add metadata columns
        source = os.path.basename(s_file),                      # Source file name          
        ticker = os.path.basename(s_file).replace('.csv', ''),  # Ticker symbol
        Date = lambda x: pd.to_datetime(x['Date'])              # Convert 'Date' column to datetime
    )
    dt_list.append(dt)                                              # Append to list 
stock_prices = pd.concat(dt_list, axis = 0, ignore_index = True)    # Merge all DataFrames into one



2025-09-26 15:26:25,757, 830353615.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks/HCM.csv
2025-09-26 15:26:25,773, 830353615.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks/ATHM.csv
2025-09-26 15:26:25,783, 830353615.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks/RILYP.csv
2025-09-26 15:26:25,791, 830353615.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks/ALDX.csv
2025-09-26 15:26:25,797, 830353615.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks/JHG.csv
2025-09-26 15:26:25,801, 830353615.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks/CHRS.csv
2025-09-26 15:26:25,811, 830353615.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks/BKTI.csv
2025-09-26 15:26:25,830, 830353615.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks/REFR.csv
2025-09-26 15:26:25,849, 830353615.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks/KEN.csv
2025-09-26 15:26:25,857, 83035

Verify the structure of the `stock_prices` data:

In [4]:
stock_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 230273 entries, 0 to 230272
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   Date       230273 non-null  datetime64[ns]
 1   Open       230260 non-null  float64       
 2   High       230260 non-null  float64       
 3   Low        230260 non-null  float64       
 4   Close      230260 non-null  float64       
 5   Adj Close  230260 non-null  float64       
 6   Volume     230260 non-null  float64       
 7   source     230273 non-null  object        
 8   ticker     230273 non-null  object        
dtypes: datetime64[ns](1), float64(6), object(2)
memory usage: 15.8+ MB


We can subset our ticker data set using standard indexing techniques. A good reference for this type of data manipulation is Panda's [Documentation](https://pandas.pydata.org/docs/user_guide/indexing.html#indexing-and-selecting-data) and [Cookbook](https://pandas.pydata.org/docs/user_guide/cookbook.html#cookbook-selection).

From the subset data frame, select one column and convert to list.

In [5]:
# Extracts the unique ticker symbols from the stock_prices DataFrame and stores them in a Python list called select_tickers.
select_tickers = stock_prices['ticker'].unique().tolist()
select_tickers

['HCM',
 'ATHM',
 'RILYP',
 'ALDX',
 'JHG',
 'CHRS',
 'BKTI',
 'REFR',
 'KEN',
 'MDRRP',
 'GRF',
 'KRC',
 'MICT',
 'AMPY',
 'A',
 'WRLSU',
 'PNR',
 'TERP',
 'TMSR',
 'TDJ',
 'CIK',
 'SMBC',
 'CNSP',
 'BLK',
 'CLNC',
 'AOSL',
 'LYV',
 'IX',
 'AWRE',
 'KNX',
 'TRXC',
 'CVCY',
 'UVE',
 'HCKT',
 'TFSL',
 'ACB',
 'AXP',
 'GLG',
 'MFO',
 'PCB',
 'IPDN',
 'DTW',
 'IQV',
 'EVRI',
 'LIVE',
 'KZR',
 'SO',
 'INFY',
 'BSE',
 'PSN',
 'TNAV',
 'APTX',
 'LAZY',
 'DK',
 'ZEUS',
 'IRBT',
 'AMAT',
 'PSB',
 'CTSO',
 'UEC']

# Storing Data in CSV



+ We have some data. How do we store it?
+ We can compare two options, CSV and Parqruet, by measuring their performance:

    - Time to save.
    - Space required.

In [6]:

# Utility function to get directory size.
# The function walks through a directory tree recursively and sums the sizes of every file it finds, returning the total size in bytes.

def get_dir_size(path='.'):
    '''Returns the total size of files contained in path.'''
    total = 0
    with os.scandir(path) as it:
        for entry in it:
            if entry.is_file():
                total += entry.stat().st_size
            elif entry.is_dir():
                total += get_dir_size(entry.path)
    return total

In [7]:
import time
import shutil

In [8]:
# Setup the directory structure 
temp = os.getenv("TEMP_DATA")
csv_dir = os.path.join(temp, "csv")
shutil.rmtree(csv_dir, ignore_errors=True)
stock_csv = os.path.join(csv_dir, "stock_px.csv")
os.makedirs(csv_dir, exist_ok=True)

In [9]:

start = time.time()
stock_prices.to_csv(stock_csv, index = False)
end = time.time()

_logs.info(f'Writing data ({stock_prices.shape}) to csv took {end - start} seconds.')
_logs.info(f'CSV file size { os.path.getsize(stock_csv)*1e-6 } MB')

2025-09-26 15:27:40,813, 473192941.py, 5, INFO, Writing data ((230273, 9)) to csv took 2.109884023666382 seconds.
2025-09-26 15:27:40,814, 473192941.py, 6, INFO, CSV file size 25.944855999999998 MB


## Save Data to Parquet

### Dask 

We can work with with large data sets and parquet files. In fact, recent versions of pandas support pyarrow data types and future versions will require a pyarrow backend. The pyarrow library is an interface between Python and the Appache Arrow project. The [parquet data format](https://parquet.apache.org/) and [Arrow](https://arrow.apache.org/docs/python/parquet.html) are projects of the Apache Foundation.

However, Dask is much more than an interface to Arrow: Dask provides parallel and distributed computing on pandas-like dataframes. It is also relatively easy to use, bridging a gap between pandas and Spark. 

In [ ]:
# Load Dask DataFrame API and prepare a clean directory parquet_dir
# that will be used to store Parquet files.

import dask.dataframe as dd

parquet_dir = os.path.join(temp, "parquet")
shutil.rmtree(parquet_dir, ignore_errors=True)
os.makedirs(parquet_dir, exist_ok=True)

/opt/anaconda3/envs/dsi_participant/lib/python3.12/site-packages/dask/dataframe/_pyarrow_compat.py:17: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 13.0.0. Please consider upgrading.
  warnings.warn(


In [11]:
# Convert the Pandas DataFrame to a Dask DataFrame with a number of partitions equal to the number of unique tickers.
# Each partition can be processed in parallel. This is particularly useful for large datasets that do not fit into memory.
px_dd = dd.from_pandas(stock_prices, npartitions = len(select_tickers))

# Write the Dask DataFrame to Parquet files using the PyArrow engine.
# Count the time (epoch) taken for this operation and log the size of the resulting Parquet files.
start = time.time()

# Write the Dask DataFrame to Parquet files using the PyArrow engine.
px_dd.to_parquet(parquet_dir, engine = "pyarrow")        
end = time.time()

# Log the time taken and size of the Parquet files
_logs.info(f'Writing dd ({stock_prices.shape}) to parquet took {end - start} seconds.')
_logs.info(f'Parquet file size { get_dir_size(parquet_dir)*1e-6 } MB')

2025-09-26 15:46:54,715, 916763503.py, 14, INFO, Writing dd ((230273, 9)) to parquet took 1.078935146331787 seconds.
2025-09-26 15:46:54,717, 916763503.py, 15, INFO, Parquet file size 8.936945 MB


### Parquet files and Dask Dataframes

+ Parquet files are immutable: once written, they cannot be modified.
+ Dask DataFrames are a useful implementation to manipulate data stored in parquets.
+ Parquet and Dask are not the same: parquet is a file format that can be accessed by many applications and programming languages (Python, R, PowerBI, etc.), while Dask is a package in Python to work with large datasets using distributed computation.
+ **Dask is not for everything** (see [Dask DataFrames Best Practices](https://docs.dask.org/en/stable/dataframe-best-practices.html)). 

    - Consider cases suchas small to large joins, where the small dataframe fits in memory, but the large one does not. 
    - If possible, use pandas: reduce, then use pandas.
    - Pandas performance tips apply to Dask.
    - Use the index: it is beneficial to have a well-defined index in Dask DataFrames, as it may speed up searching (filtering) the data. A one-dimensional index is allowed.
    - Avoid (or minimize) full-data shuffling: indexing is an expensive operations. 
    - Some joins are more expensive than others. 

        * Not expensive:

            - Join a Dask DataFrame with a pandas DataFrame.
            - Join a Dask DataFrame with another Dask DataFrame of a single partition.
            - Join Dask DataFrames along their indexes.

        * Expensive:

            - Join Dask DataFrames along columns that are not their index.


# How do we store prices?

+ We can store our data as a single blob. This can be difficult to maintain, especially because parquet files are immutable.
+ Strategy: organize data files by ticker and date. Update only latest month.



In [12]:
# CLean up before start
PRICE_DATA = os.getenv("PRICE_DATA")
import shutil
# Remove the PRICE_DATA directory if it exists to ensure a clean state before starting any data processing tasks.
if os.path.exists(PRICE_DATA):
    shutil.rmtree(PRICE_DATA)

In [13]:
stock_prices.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker'],
      dtype='object')

### The Loop

- **Creates a folder for each stock ticker** (e.g., `AAPL`, `MSFT`).
- **Inside each ticker folder**, creates a Parquet dataset for each year (e.g., `AAPL_2023`, `AAPL_2024`).
- **Saves that year’s data**—only rows for that ticker and year—into the corresponding Parquet folder.

### Example Structure

![image.png](./images/partitioned_parquet_layout.png)
### Notes

- **`PRICE_DATA/`** – Base directory for all partitioned data.  
- **`A/` and `ACB/`** – One folder per ticker symbol.  
- **`A_1999`, `ACB_2004`, etc.** – Subfolders for each year of that ticker’s data.  
- **`part.*.parquet`** – The actual Parquet data files written by Dask, usually one file per partition.

In [ ]:
# This loop partitions and saves the dataset so that:
#   Per Ticker: Each stock symbol gets its own folder.
#   Per Year: Within that folder, data is split into one Parquet dataset per year.
#  This makes it easy to load just the slices you need (e.g., a specific ticker and year)
#  without scanning a massive single file.

for ticker in stock_prices['ticker'].unique():
    ticker_dt = stock_prices[stock_prices['ticker'] == ticker]
    ticker_dt = ticker_dt.assign(Year = ticker_dt.Date.dt.year)
    for yr in ticker_dt['Year'].unique():
        yr_dd = dd.from_pandas(ticker_dt[ticker_dt['Year'] == yr],2)
        yr_path = os.path.join(PRICE_DATA, ticker, f"{ticker}_{yr}")
        os.makedirs(os.path.dirname(yr_path), exist_ok=True)
        yr_dd.to_parquet(yr_path, engine = "pyarrow")
    

Why would we want to store data this way?

+ Easier to maintain. We do not update old data, only recent data.
+ We can also access all files as follows.

# Load, Transform and Save 

## Load

+ Parquet files can be read individually or as a collection.
+ `dd.read_parquet()` can take a list (collection) of files as input.
+ Use `glob` to get the collection of files.

In [15]:
from glob import glob

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

In [17]:
# parquet_files
# Show the structure of dd_px
dd_px


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=60,,,,,,,,,
A,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32
ACB,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...


## Transform

+ This transformation step will create a *Features* data set. In our case, features will be stock returns (we obtained prices).
+ Dask dataframes work like pandas dataframes: in particular, we can perform groupby and apply operations.
+ Notice the use of [an anonymous (lambda) function](https://realpython.com/python-lambda/) in the apply statement.

In [18]:
stock_prices


,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker
0,2016-03-16,13.50,13.50,13.50,13.50,13.50,0.0,HCM.csv,HCM
1,2016-03-17,13.50,14.20,13.02,13.40,13.40,2254400.0,HCM.csv,HCM
2,2016-03-18,13.42,13.50,13.30,13.50,13.50,299500.0,HCM.csv,HCM
3,2016-03-21,13.50,13.75,13.35,13.50,13.50,137800.0,HCM.csv,HCM
4,2016-03-22,13.21,13.50,13.21,13.38,13.38,150100.0,HCM.csv,HCM
...,...,...,...,...,...,...,...,...,...
230268,2020-03-26,0.59,0.60,0.54,0.57,0.57,1036000.0,UEC.csv,UEC
230269,2020-03-27,0.55,0.55,0.50,0.51,0.51,1200100.0,UEC.csv,UEC
230270,2020-03-30,0.51,0.54,0.50,0.53,0.53,1100300.0,UEC.csv,UEC
230271,2020-03-31,0.51,0.57,0.51,0.56,0.56,1296900.0,UEC.csv,UEC


In [21]:
# Perform a groupby operation on 'ticker' and apply a function to create a new column 'Close_lag_1' that contains the previous day's closing price for each ticker.
dd_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)

/var/folders/5h/h1f0cx75553g3c45m_p5b0k80000gn/T/ipykernel_4476/2347871401.py:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby('ticker', group_keys=False).apply(


In [22]:
dd_shift

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1
npartitions=60,,,,,,,,,,
A,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64
ACB,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...


In [23]:
dd_rets = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)

In [24]:
dd_rets


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns
npartitions=60,,,,,,,,,,,
A,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64
ACB,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...,...


## Lazy Exection

What does `dd_rets` contain?

In [25]:
# Compute the final result and show the sample rows
dd_rets.compute()


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns
ticker,,,,,,,,,,,
A,2015-07-06,39.110001,39.660000,39.080002,39.360001,37.599323,2509400.0,A.csv,2015,NaN,NaN
A,2015-07-07,39.529999,39.790001,39.139999,39.790001,38.010090,2927800.0,A.csv,2015,39.360001,0.010925
A,2015-07-08,39.480000,39.480000,38.709999,38.750000,37.016602,3391000.0,A.csv,2015,39.790001,-0.026137
A,2015-07-09,39.270000,39.330002,38.910000,38.919998,37.179005,2237900.0,A.csv,2015,38.750000,0.004387
A,2015-07-10,39.290001,39.439999,39.160000,39.400002,37.637531,2107100.0,A.csv,2015,38.919998,0.012333
...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,2003-12-24,6.950000,7.000000,6.880000,6.990000,6.401954,10700.0,ZEUS.csv,2003,6.870000,0.017467
ZEUS,2003-12-26,7.000000,7.750000,6.800000,7.630000,6.988110,179300.0,ZEUS.csv,2003,6.990000,0.091559
ZEUS,2003-12-29,7.700000,8.500000,7.670000,8.200000,7.510161,221700.0,ZEUS.csv,2003,7.630000,0.074705


+ Dask is a lazy execution framework: commands will not execute until they are required. 
+ To trigger an execution in dask use `.compute()`.

In [26]:
dd_rets.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns
ticker,,,,,,,,,,,
A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,-0.082386
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,0.089783
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,-0.090909
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,0.026563
...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,2020-03-26,9.610000,9.940000,9.260000,9.590000,9.590000,60500.0,ZEUS.csv,2020,9.670000,-0.008273
ZEUS,2020-03-27,9.330000,9.330000,8.700000,8.700000,8.700000,52900.0,ZEUS.csv,2020,9.590000,-0.092805
ZEUS,2020-03-30,8.810000,9.760000,8.700000,9.680000,9.680000,73700.0,ZEUS.csv,2020,8.700000,0.112644


## Save

+ Apply transformations to calculate daily returns
+ Store the enriched data, the silver dataset, in a new directory.
+ Should we keep the same namespace? All columns?

In [27]:
# CLean up before save
FEATURES_DATA = os.getenv("FEATURES_DATA")
if os.path.exists(FEATURES_DATA):
    shutil.rmtree(FEATURES_DATA)
dd_rets.to_parquet(FEATURES_DATA, overwrite = True)

# Optional: from Jupyter to Command Line

+ We have drafted our code in a Jupyter Notebook. 
+ Finalized code should be written in Python modules.

## Object Oriented vs Functional Programming

+ We can use classes to keep parameters and functions together.
+ We *could* use Object Oriented Programming, but parallelization of data manipulation and modelling tasks benefit from *Functional Programming*.
+ An Idea: 

    - [Data Oriented Programming](https://blog.klipse.tech/dop/2022/06/22/principles-of-dop.html).
    - Use the class to bundle together parameters and functions.
    - Use stateless operations and treat all data objects as immutable (we do not modify them, we overwrite them).
    - Take advantage of [`@staticmethod`](https://realpython.com/instance-class-and-static-methods-demystified/).

The code is in `./05_src/stock_prices/data_manager.py`.

Our original design was:

![](./images/02_target_pipeline_manager.png)



In [26]:
from stock_prices.data_manager import DataManager
dm = DataManager()

In [ ]:
# Show object dm
dm

In [ ]:
# Show the price directory
dm.price_dir

'../../05_src/data/prices/'

Download all prices.

In [29]:
dm.process_sample_files()

2025-09-25 18:53:16,685, data_manager.py, 53, INFO, Processing sample of tickers
2025-09-25 18:53:16,694, data_manager.py, 64, INFO, Getting file list from ../../05_src/data/prices_csv/
2025-09-25 18:53:16,746, data_manager.py, 66, INFO, Found 5884 files in ../../05_src/data/prices/
2025-09-25 18:53:16,747, data_manager.py, 74, INFO, Selecting sample of files
2025-09-25 18:53:16,749, data_manager.py, 80, INFO, Selected 30 files
2025-09-25 18:53:16,750, data_manager.py, 90, INFO, Processing file ../../05_src/data/prices_csv/stocks/FSLR.csv
2025-09-25 18:53:16,751, data_manager.py, 118, INFO, Reading file: ../../05_src/data/prices_csv/stocks/FSLR.csv
2025-09-25 18:53:16,859, data_manager.py, 101, INFO, Saving data by year
2025-09-25 18:53:17,413, data_manager.py, 90, INFO, Processing file ../../05_src/data/prices_csv/stocks/WYY.csv
2025-09-25 18:53:17,413, data_manager.py, 118, INFO, Reading file: ../../05_src/data/prices_csv/stocks/WYY.csv
2025-09-25 18:53:17,440, data_manager.py, 101, 

Finally, add features to the data set and save to a *feature store*.

In [30]:
dm.featurize()

2025-09-25 18:53:54,323, data_manager.py, 131, INFO, Creating features data.
2025-09-25 18:53:54,325, data_manager.py, 141, INFO, Loading price data from ../../05_src/data/prices/
2025-09-25 18:53:54,746, data_manager.py, 150, INFO, Creating features
/Users/vincent/production/01_materials/labs/../../05_src/stock_prices/data_manager.py:154: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  .apply(
2025-09-25 18:53:54,757, data_manager.py, 175, INFO, Saving features to ../../05_src/data/features/stock_features
